# Match recordings to rukus on a Colab GPU

Runs `scripts/match-audio.py` on Colab's GPU instead of a local card.

**Nothing is uploaded.** The audio is already in the public GitHub repo, so Colab clones
it directly inside Google's network rather than pulling it off your machine.

Worth it because VRAM is the local bottleneck: a 4GB laptop card only fits `medium` at
batch 4, while a Colab T4 has 16GB and fits batch 16-24. Local measured ~8s per
recording, so ~70 min for all 528.

Set **Runtime -> Change runtime type -> T4 GPU** before running.


## 1. Confirm a GPU is actually attached


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# No output here means the runtime is CPU-only: Runtime -> Change runtime type -> T4 GPU.


## 2. Dependencies

`node` is needed for `scripts/dump-context.js`, which hands data.js/verses.js and the
bookmark to the Python matcher. Colab images do not always ship it.


In [ ]:
!pip -q install faster-whisper
!which node || (apt-get -qq update && apt-get -qq install -y nodejs)
!node --version


## 3. Clone the repo

`--depth 1` skips the history; only the current tree is needed, and history roughly
doubles the transfer for a repo this size.


In [ ]:
!git clone --depth 1 --branch v4 https://github.com/mohsingdp-ai/Marifatul-Quran.git repo
%cd repo
!du -sh audio && ls audio | head -5


## 4. Run the matcher

`--batch 16` is the point of the exercise -- it is what the local 4GB card cannot do.
Drop to 8 if you hit an out-of-memory error; the script also steps down on its own.

Start with a para whose answers are already known (1, 3 and 4 are verified correct) to
confirm the setup agrees before trusting a full run.


In [ ]:
!python scripts/match-audio.py --para 3 --model medium --device cuda --batch 16


## 5. Everything, once the check above looks right

Roughly 20-25 min on a T4. Colab drops idle sessions, so keep the tab open.


In [ ]:
!python scripts/match-audio.py --all --model medium --device cuda --batch 16 \
    2>/dev/null | tee /content/match-audio-results.txt
print(open('/content/match-audio-results.txt').read()[-3000:])


## 6. Bring the results back

Transcripts are the valuable part -- they are what lets a flag be checked by reading it
rather than trusting the verdict. Small enough to download; the audio stays put.


In [ ]:
!cd /content/repo && zip -qr /content/asr-cache.zip .cache/asr
from google.colab import files
files.download('/content/match-audio-results.txt')
files.download('/content/asr-cache.zip')


## Reading the output

- `ok` -- the claimed ruku is the best match, or close to it
- `weak` -- not enough evidence either way, usually a very short ruku. Not an accusation.
- `SUSPECT` -- the claim ranks far down. **Read the transcript before believing it.**

Every SUSPECT raised so far has been a false positive with data.js correct, so treat the
verdict as a prompt to look, never as a finding. Drop the cache zip into `.cache/asr/`
locally and the transcripts are there to read.
